In [1]:
import json

In [2]:
# split = 'val'
# split = 'train'
split = 'test'

# mode = 'cv'
mode = 'cvi'

inf = True

directory = f'/home/comp/24483737/fc/data/RAWFC/{split}'

In [3]:
def load_all_json_dicts_from_dir(dir_path):
    all_dicts = []
    for filename in os.listdir(dir_path):
        if filename.endswith('.json'):
            file_path = os.path.join(dir_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                    if isinstance(data, dict):
                        all_dicts.append(data)
                    else:
                        print(f"Warning: {filename} 不是一个字典，跳过")
                except json.JSONDecodeError as e:
                    print(f"Error decoding {filename}: {e}")
    return all_dicts

json_dicts = load_all_json_dicts_from_dir(directory)

print(f"共加载 {len(json_dicts)} 个字典")

共加载 200 个字典


In [4]:
json_dicts[0]

{'event_id': '101929',
 'claim': 'On 1 September 2017, a new law will come into force in Texas, making it harder for consumers to get paid for property insurance claims related to weather.',
 'original_label': 'mixture',
 'label': 'half',
 'explain': 'On 27 August 2017, as the full damage to Texas from a storm of historic proportions was not yet fully known, a number of social media posts began to spread, alerting readers to a change in Texas law around insurance claims in the event of weather damage.\xa0One Facebook user\xa0posted\xa0the following:Former Texas state representative Glen Maxey wrote:And on 29 August, Rep. Joaquín Castro encouraged Texans affected by Tropical Storm Harvey to file their claims before 1 September:Texas House Bill 1774 does indeed introduce changes to the way property damage insurance claims are dealt with in the state, but only in the event of a litigated dispute over a claim. Most insurance claims are settled out of court, and so the potential benefit in 

In [5]:
from collections import Counter

label_counter = Counter()

for d in json_dicts:
    label = d.get('label')
    if label is not None:
        label_counter[label] += 1
    else:
        print("Warning: 某个字典中没有 'label' 字段，已跳过")

for label, count in label_counter.items():
    print(f"{label}: {count}")


half: 67
true: 67
false: 66


In [6]:
def clean_wi_inf(raw_list, mode):
    conv_list = []
    for event in raw_list:
        conv = {}
        conv['id'] = event['event_id']
        
        claim = event['claim']
        label = event['label']
        inf = event['explain']

        evidences = ''
        evidence_id = 1 # 
        reports_list = event['reports']
        for rep_dict in reports_list:
            tokenized_list = rep_dict['tokenized']
            for rep in tokenized_list:
                if rep["is_evidence"] != 0: 
                    evidences += (f"({evidence_id}) {rep['sent']}")  
                    # evidences += rep['sent']
                    evidence_id += 1

        if len(evidences) == 0:
            continue
        
        utter_list = []
        if mode == 'cev':
            utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
        else:
            utter_list.append({"from": "human", "value": f"Claim: {claim}"})
        # utter_list.append({"from": "gpt", "value": f"Verdict: {label}. Reasoning: {inf}"})
        utter_list.append({"from": "gpt", "value": f"Verdict: {label}. Explanation: {inf}"})
        conv['conversations'] = utter_list
        conv_list.append(conv)
    return conv_list
   

In [7]:
def clean(raw_list):
    conv_list = []
    for event in raw_list:
        conv = {}
        conv['id'] = event['event_id']
        
        claim = event['claim']
        # label = event['label']

        # 交叉指令跟随
        raw_label = event['label']
        if raw_label == 'half':
            label = 'HALF-TRUE'
        elif raw_label == 'false':
            label = 'FALSE'
        else:
            label = 'TRUE'

        evidences = ''
        evidence_id = 1 # 
        reports_list = event['reports']
        for rep_dict in reports_list:
            tokenized_list = rep_dict['tokenized']
            for rep in tokenized_list:
                if rep["is_evidence"] != 0: # 优化下限
                    evidences += (f"({evidence_id}) {rep['sent']}") # 
                    # evidences += rep['sent']
                    evidence_id += 1

        if len(evidences) == 0:
            continue
        
        utter_list = []
        # utter_list.append({"from": "human", "value": f"Claim: {claim}"})
        utter_list.append({"from": "human", "value": f"Claim: {claim}\nEvidences: {evidences}"})
        utter_list.append({"from": "gpt", "value": f"Verdict: {label}"})
        conv['conversations'] = utter_list
        conv_list.append(conv)
    return conv_list
   

In [8]:
if inf:
    data = clean_wi_inf(json_dicts, mode)
else:
    data = clean(json_dicts)
len(data)

200

In [9]:
data[0]

{'id': '101929',
 'conversations': [{'from': 'human',
   'value': 'Claim: On 1 September 2017, a new law will come into force in Texas, making it harder for consumers to get paid for property insurance claims related to weather.'},
  {'from': 'gpt',
   'value': 'Verdict: half. Explanation: On 27 August 2017, as the full damage to Texas from a storm of historic proportions was not yet fully known, a number of social media posts began to spread, alerting readers to a change in Texas law around insurance claims in the event of weather damage.\xa0One Facebook user\xa0posted\xa0the following:Former Texas state representative Glen Maxey wrote:And on 29 August, Rep. Joaquín Castro encouraged Texans affected by Tropical Storm Harvey to file their claims before 1 September:Texas House Bill 1774 does indeed introduce changes to the way property damage insurance claims are dealt with in the state, but only in the event of a litigated dispute over a claim. Most insurance claims are settled out of 

In [10]:
label_counter = Counter()
for d in data:
    label = d.get('conversations')[1].get('value')
    if label is not None:
        label_counter[label] += 1
    else:
        print("Warning: 某个字典中没有 'label' 字段，已跳过")



In [13]:
with open(f'./rawfc-post/small-original-half/{split}_rawfc_{mode}.json', 'w', encoding="utf-8" ) as file:
    json.dump(data, file, indent=2, ensure_ascii=False)